# Week 1 — Tour of an anonymized claims sample
## GLUCOREN: reading a claims row like a commercial analyst

**Applied Data Science and AI in Pharma Commercial · Fall 2026**

This is a *tour*, not an analysis. In ten minutes we look at what claims data actually contains, and connect every column to a stakeholder from tonight's map. Next week (Week 2) we build a real cohort and assign lines of therapy on a much larger synthetic dataset.

The data is **synthetic**: 60 fictional patients treated for type 2 diabetes in the New York area, prescribed the fictional brand GLUCOREN (Aspen Therapeutics), its fictional competitor CARDIVEX, or generic metformin. No real patients, prescribers or plans are represented; NPIs are made up (but Luhn-valid in format).

Files (all CSV, in this folder):

| file | grain | what it represents |
|---|---|---|
| `glucoren_sample_patients.csv` | one row per patient | de-identified patient token, demographics, plan |
| `glucoren_sample_medical_claims.csv` | one row per visit | diagnosis (ICD-10) and procedure (CPT) codes, rendering NPI |
| `glucoren_sample_pharmacy_claims.csv` | one row per adjudication event | NDC, days' supply, status (paid / rejected / reversed), who paid what |
| `glucoren_sample_drug_master.csv` | one row per NDC | NDC → drug → molecule → brand → labeler → WAC |
| `glucoren_sample_providers.csv` | one row per NPI | specialty, HCO/IDN affiliation |
| `glucoren_sample_plans.csv` | one row per plan | payer type, PBM, GLUCOREN formulary tier, prior-auth flag |


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
patients  = pd.read_csv("data/glucoren_sample_patients.csv")
medical   = pd.read_csv("data/glucoren_sample_medical_claims.csv", parse_dates=["service_date"])
pharmacy  = pd.read_csv("data/glucoren_sample_pharmacy_claims.csv", parse_dates=["service_date", "reversal_date"])
drugs     = pd.read_csv("data/glucoren_sample_drug_master.csv")
providers = pd.read_csv("data/glucoren_sample_providers.csv")
plans     = pd.read_csv("data/glucoren_sample_plans.csv")
print(f"patients {len(patients)} · medical claims {len(medical)} · pharmacy claims {len(pharmacy)}")

patients 60 · medical claims 126 · pharmacy claims 162


## 1. The patient — and why you only ever see a token
`patient_id` is a **token**, not an identity. Real vendors replace name, address and date of birth with a one-way hash so that claims for the same person can be linked over time (longitudinal data) without revealing who they are (HIPAA de-identification). Age comes as a band or a birth year; ZIP-3 at best. Re-identification risk is the reason Week 2 spends time on data governance.

In [3]:
patients.head(8)

,patient_id,birth_year,sex,state,plan_id,age_band
0,P00001,1972,M,NY,PL-MED-01,50-64
1,P00002,1955,M,NY,PL-MED-01,65+
2,P00003,1981,F,CT,PL-COM-01,<50
3,P00004,1964,F,NY,PL-MED-01,50-64
4,P00005,1948,F,NJ,PL-MCD-01,65+
5,P00006,1976,F,NY,PL-MED-01,50-64
6,P00007,1969,M,NJ,PL-MED-01,50-64
7,P00008,1982,F,NJ,PL-COM-02,<50


## 2. The medical claim — where the diagnosis lives
A medical claim is what the **provider (HCP/HCO)** bills to the **payer** for a visit or procedure. Three codes matter tonight:

- `dx1_icd10` — the diagnosis. `E11.*` is type 2 diabetes; the characters after the decimal describe complications (E11.65 with hyperglycemia, E11.22 with kidney disease).
- `cpt_code` — the service (99213/99214/99215 are office visits of increasing complexity; 83036 is an HbA1c lab test).
- `rendering_npi` — the individual clinician; `billing_hco_id` — the organization paid. Same doctor, different organizations → different economics (an IDN clinic may be 340B-eligible).

In [4]:
medical.head(8)

,claim_id,patient_id,service_date,rendering_npi,billing_hco_id,place_of_service,dx1_icd10,dx2_icd10,cpt_code,cpt_code_2,plan_id,allowed_amount
0,M000002,P00001,2026-02-01,1659873021,HCO-002,11,E11.65,NaN,99214,83036.0,PL-MED-01,114.03
1,M000003,P00001,2026-05-15,1487265930,HCO-005,11,E11.51,I10,99213,83036.0,PL-MED-01,236.79
2,M000001,P00001,2026-07-03,1568923410,HCO-004,11,E11.9,NaN,99213,83036.0,PL-MED-01,170.56
3,M000004,P00002,2026-02-03,1932145067,HCO-002,11,E11.65,NaN,99215,NaN,PL-MED-01,275.93
4,M000006,P00002,2026-03-27,1720394856,HCO-003,11,E11.65,NaN,99214,NaN,PL-MED-01,231.02
5,M000005,P00002,2026-03-29,1720394856,HCO-003,11,E11.9,NaN,99214,83036.0,PL-MED-01,141.54
6,M000007,P00003,2026-04-18,1801234569,HCO-006,11,E11.65,NaN,99213,83036.0,PL-COM-01,132.26
7,M000008,P00003,2026-05-18,1234567893,HCO-001,11,E11.65,NaN,99214,83036.0,PL-COM-01,184.48


In [5]:
# Which diagnoses, which specialties?
m = medical.merge(providers, left_on="rendering_npi", right_on="prescriber_npi")
print(m.dx1_icd10.value_counts(), end="\n\n")
print(m.specialty.value_counts())

dx1_icd10
E11.9     62
E11.65    41
E11.51    16
E11.22     7
Name: count, dtype: int64

specialty
Endocrinology         36
Family Medicine       31
Internal Medicine     23
Nurse Practitioner    20
Cardiology            16
Name: count, dtype: int64


## 3. The pharmacy claim — the prescription's whole life in one table
A pharmacy claim is adjudicated **in real time** by the **PBM** when the pharmacist scans the prescription. That single transaction encodes almost every stakeholder:

| column | stakeholder / concept |
|---|---|
| `ndc`, `drug_name`, `brand` | the **manufacturer's** product (join to drug master for molecule, labeler, WAC) |
| `prescriber_npi` | the **HCP** who chose it (join to providers for specialty, HCO/IDN) |
| `plan_id`, `payer_type`, `pbm` | the **payer** and the **PBM** — the access gate |
| `claim_status`, `reject_code` | utilization management in action: `75` = prior authorization required |
| `days_supply`, `service_date` | patient behavior: refills, gaps, persistence |
| `plan_paid`, `patient_paid`, `copay_card_paid` | the money flow at the counter, including the **manufacturer's** copay card |
| `pharmacy_type` | the channel (retail / mail — often PBM-owned) |
| `reversal_date` | abandonment: the claim was adjudicated, the patient never picked it up |

In [6]:
pharmacy.head(12)

,claim_id,patient_id,service_date,ndc,drug_name,brand,prescriber_npi,plan_id,payer_type,pbm,pharmacy_type,days_supply,claim_status,reject_code,ingredient_cost_submitted,plan_paid,patient_paid,copay_card_paid,reversal_date
0,R000001,P00001,2026-02-03,00042-1010-30,GLUCOREN 10 mg tablet,GLUCOREN,1659873021,PL-MED-01,Medicare Part D,PBM-C (Optum-like),Retail,0,REJECTED,75.0,NaN,NaN,NaN,NaN,NaT
1,R000002,P00001,2026-02-12,00042-1010-30,GLUCOREN 10 mg tablet,GLUCOREN,1659873021,PL-MED-01,Medicare Part D,PBM-C (Optum-like),Retail,30,PAID,NaN,511.1,464.1,47.0,0.0,NaT
2,R000003,P00001,2026-03-19,00042-1010-30,GLUCOREN 10 mg tablet,GLUCOREN,1659873021,PL-MED-01,Medicare Part D,PBM-C (Optum-like),Retail,30,PAID,NaN,511.1,464.1,47.0,0.0,NaT
3,R000004,P00001,2026-04-18,00042-1010-30,GLUCOREN 10 mg tablet,GLUCOREN,1659873021,PL-MED-01,Medicare Part D,PBM-C (Optum-like),Retail,30,PAID,NaN,511.1,464.1,47.0,0.0,NaT
4,R000005,P00001,2026-05-29,00042-1010-30,GLUCOREN 10 mg tablet,GLUCOREN,1659873021,PL-MED-01,Medicare Part D,PBM-C (Optum-like),Retail,30,PAID,NaN,511.1,464.1,47.0,0.0,NaT
5,R000006,P00002,2026-02-03,00042-1010-90,GLUCOREN 10 mg tablet (90 ct),GLUCOREN,1932145067,PL-MED-01,Medicare Part D,PBM-C (Optum-like),Retail,0,REJECTED,75.0,NaN,NaN,NaN,NaN,NaT
6,R000007,P00002,2026-02-08,00042-1010-90,GLUCOREN 10 mg tablet (90 ct),GLUCOREN,1932145067,PL-MED-01,Medicare Part D,PBM-C (Optum-like),Mail,90,PAID,NaN,1530.3,1483.3,47.0,0.0,NaT
7,R000008,P00002,2026-05-10,00042-1010-90,GLUCOREN 10 mg tablet (90 ct),GLUCOREN,1932145067,PL-MED-01,Medicare Part D,PBM-C (Optum-like),Mail,90,PAID,NaN,1530.3,1483.3,47.0,0.0,NaT
8,R000009,P00002,2026-08-06,00042-1010-90,GLUCOREN 10 mg tablet (90 ct),GLUCOREN,1932145067,PL-MED-01,Medicare Part D,PBM-C (Optum-like),Mail,90,PAID,NaN,1530.3,1483.3,47.0,0.0,NaT
9,R000010,P00003,2026-04-19,00042-1010-30,GLUCOREN 10 mg tablet,GLUCOREN,1801234569,PL-COM-01,Commercial,PBM-A (CVS Caremark-like),Retail,30,PAID,NaN,511.1,466.1,25.0,20.0,NaT


### 3a. Adjudication outcomes — rejections and reversals are where access analytics starts

In [7]:
print(pharmacy.claim_status.value_counts(), end="\n\n")
rej = pharmacy[pharmacy.claim_status == "REJECTED"].merge(plans, on="plan_id", suffixes=("", "_plan"))
print("Rejected claims by plan / prior-auth flag:")
print(rej.groupby(["plan_id", "glucoren_prior_auth"]).size())

claim_status
PAID        152
REJECTED      7
REVERSED      3
Name: count, dtype: int64

Rejected claims by plan / prior-auth flag:
plan_id    glucoren_prior_auth
PL-COM-02  Yes                    1
PL-MED-01  Yes                    6
dtype: int64


In [8]:
rev = pharmacy[pharmacy.claim_status == "REVERSED"]
print("Reversed (abandoned) fills — note the patient_paid amount that triggered them:")
rev[["patient_id", "service_date", "brand", "payer_type", "patient_paid", "copay_card_paid", "reversal_date"]]

Reversed (abandoned) fills — note the patient_paid amount that triggered them:


,patient_id,service_date,brand,payer_type,patient_paid,copay_card_paid,reversal_date
39,P00015,2026-01-20,GLUCOREN,Cash / uninsured,1530.3,0.0,2026-02-02
89,P00031,2026-03-11,CARDIVEX,Cash / uninsured,535.6,0.0,2026-03-24
161,P00060,2026-07-11,GLUCOREN,Cash / uninsured,1530.3,0.0,2026-07-24


### 3b. Counting prescriptions: TRx and who paid
`TRx` = paid claims. A normalized TRx converts a 90-day fill into three 30-day equivalents. Notice how much of the brand's revenue is paid by the plan versus the patient — and that the copay card (paid by the manufacturer) is itself a gross-to-net deduction.

In [9]:
paid = pharmacy[pharmacy.claim_status == "PAID"].copy()
paid["trx_30d"] = paid.days_supply / 30
summary = (paid.groupby("brand")
               .agg(claims=("claim_id", "count"), trx_normalized=("trx_30d", "sum"),
                    plan_paid=("plan_paid", "sum"), patient_paid=("patient_paid", "sum"), copay_card=("copay_card_paid", "sum"))
               .round(1))
summary

,claims,trx_normalized,plan_paid,patient_paid,copay_card
brand,,,,,
CARDIVEX,45,45.0,20785.2,1235.0,2081.8
GLUCOREN,92,142.0,60637.6,6700.8,5162.8
metformin (generic),15,45.0,-9.8,75.0,0.0


In [10]:
# Patient out-of-pocket by payer type for GLUCOREN — the patient's view of formulary tier
g = paid[paid.brand == "GLUCOREN"]
g.groupby("payer_type").agg(fills=("claim_id", "count"), avg_patient_paid=("patient_paid", "mean"), avg_card=("copay_card_paid", "mean")).round(2)

,fills,avg_patient_paid,avg_card
payer_type,,,
Cash / uninsured,4,1020.7,0.00
Commercial,32,25.0,31.25
Commercial HDHP,13,25.0,320.22
Medicaid,12,3.0,0.00
Medicare Part D,31,47.0,0.00


## 4. Joining the codes — the manufacturer's view is a projection
The manufacturer never sees this table directly. It licenses a **projected** version from a data vendor and reconciles it against its own shipments. Here we do the join the vendor does: NDC → brand → labeler, NPI → specialty → HCO.

In [11]:
joined = (paid.merge(drugs[["ndc", "molecule", "labeler", "wac_per_pack"]], on="ndc")
              .merge(providers[["prescriber_npi", "specialty", "hco_name"]], on="prescriber_npi"))
joined.pivot_table(index="specialty", columns="brand", values="trx_30d", aggfunc="sum", fill_value=0).round(1)

brand,CARDIVEX,GLUCOREN,metformin (generic)
specialty,,,
Cardiology,12.0,21.0,18.0
Endocrinology,14.0,43.0,18.0
Family Medicine,13.0,35.0,9.0
Internal Medicine,0.0,17.0,0.0
Nurse Practitioner,6.0,26.0,0.0


In [12]:
# Gross sales at WAC implied by these fills vs. what was actually paid at the counter
joined["gross_at_wac"] = joined.wac_per_pack
g = joined[joined.brand == "GLUCOREN"]
print(f"GLUCOREN fills: {len(g)} · gross at WAC ${g.gross_at_wac.sum():,.0f} · plan paid ${g.plan_paid.sum():,.0f} · patient paid ${g.patient_paid.sum():,.0f} · copay card ${g.copay_card_paid.sum():,.0f}")
print("Remember: rebates to the PBM are NOT in this table. They are invoiced to the manufacturer a quarter later — the biggest gross-to-net line is invisible in claims.")

GLUCOREN fills: 92 · gross at WAC $73,840 · plan paid $60,638 · patient paid $6,701 · copay card $5,163
Remember: rebates to the PBM are NOT in this table. They are invoiced to the manufacturer a quarter later — the biggest gross-to-net line is invisible in claims.


## 5. Persistence — the first patient-journey metric
How many GLUCOREN starters are still filling 90 days later? Even on 60 patients, you can see the shape of the problem that Week 5's adherence work addresses.

In [13]:
g = paid[paid.brand == "GLUCOREN"].sort_values(["patient_id", "service_date"])
first = g.groupby("patient_id").service_date.min().rename("first_fill")
last_cov = g.assign(covered_to=g.service_date + pd.to_timedelta(g.days_supply, unit="D")).groupby("patient_id").covered_to.max().rename("covered_to")
j = pd.concat([first, last_cov], axis=1)
j["days_on_therapy"] = (j.covered_to - j.first_fill).dt.days
print(f"GLUCOREN starters: {len(j)} · still covered at day 90: {(j.days_on_therapy >= 90).sum()} ({(j.days_on_therapy >= 90).mean():.0%})")
j.days_on_therapy.describe().round(0)

GLUCOREN starters: 35 · still covered at day 90: 24 (69%)


count     35.0
mean     130.0
std       76.0
min       30.0
25%       70.0
50%      132.0
75%      184.0
max      284.0
Name: days_on_therapy, dtype: float64

## 6. Five things to notice before you leave
1. **Every stakeholder is a column.** Manufacturer (NDC/labeler), HCP (NPI), HCO/IDN (hco_id), payer & PBM (plan_id, pbm), patient (token, patient_paid), policymaker (payer_type = Medicaid / Medicare rules).
2. **The most important event is often the missing row.** A rejected claim, a reversal, or a patient who never comes back is where access and adherence analytics live.
3. **Money in claims ≠ manufacturer revenue.** Plan-paid and patient-paid sum to the pharmacy's reimbursement; rebates, chargebacks and fees are elsewhere.
4. **Codes are the joins.** ICD-10 → cohort, NDC → product, NPI → prescriber, plan_id → access. Master data quality decides whether your analysis is right.
5. **Sixty patients is a toy.** Next week: 500 DLBCL patients, medical + pharmacy claims, J-codes, and a line-of-therapy algorithm.

### Optional homework-free exploration
- Which plan has the highest GLUCOREN abandonment? Why?
- For each prescriber, compute GLUCOREN share of their branded diabetes fills. Who would you visit first, and why is that question harder than it looks?
